In [ ]:
from datetime import datetime
from itertools import chain
import pandas as pd
from bs4 import BeautifulSoup
from helpers import get_request

In [ ]:
def get_pages(num_pages: int):
    for i in range(num_pages):
        if i % 10 == 0:
            print("Getting page", i, "/", num_pages)

        try:
            response = get_request(f"https://www.ufc.com/trending/all?page={i}")
            yield response.text
        except Exception as e:
            print("Error getting page", i)
            print(e)

            yield None

In [ ]:
# I ran this on 16 Sept. 2025 with num_pages_to_get = 200, 
# and the oldest page I got was from 12 Jan. 2025.
# Adjust accordingly for the date range you need
num_pages_to_get = 200
pages = [
    page for page in get_pages(num_pages_to_get)
    if page is not None
]
print("Got", len(pages), "pages")

In [ ]:
def get_articles_data(page_content: str):
    soup = BeautifulSoup(page_content, "html.parser")
    articles = soup.find_all("a", "c-card--grid-card-trending")

    articles_data = [
        {
            "type": link.find_all("span", "c-card--grid-card-trending__info-prefix")[0].text,
            "time_ago": link.find_all("div", "field--name-datetime")[0].text,
            "headline": link.find_all("h3", "c-card--grid-card-trending__headline")[0].text.strip(),
            "link": link["href"]
        }
        for link in articles
    ]

    for data in articles_data:
        yield data

In [ ]:
flattened_articles = list(chain.from_iterable(get_articles_data(page) for page in pages))
flattened_articles

In [ ]:
articles_df = pd.DataFrame(data=flattened_articles)
articles_df

In [ ]:
filename = f"./article_links_{datetime.now().strftime('%d-%m-%y_%H-%M-%S')}.csv"
articles_df\
    .set_index("link")\
    .to_csv(filename)